# SurfaceGuard AI: Training, Tuning and Inference
Roll No: 23f1003053

## 1. Install libraries

In [1]:
!pip install -q segmentation-models-pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.6 MB/s eta 0:00:00


In [2]:
import segmentation_models_pytorch as smp
print(smp.__version__)

0.5.0


In [3]:
!pip download segmentation-models-pytorch --no-deps -d /kaggle/working/smp_wheels
!ls /kaggle/working/smp_wheels

  Using cached segmentation_models_pytorch-0.5.0-py3-none-any.whl.metadata (17 kB)
Using cached segmentation_models_pytorch-0.5.0-py3-none-any.whl (154 kB)
Saved ./smp_wheels/segmentation_models_pytorch-0.5.0-py3-none-any.whl
Successfully downloaded segmentation-models-pytorch
segmentation_models_pytorch-0.5.0-py3-none-any.whl


## 2. Weights & Biases login

In [4]:
import wandb
from kaggle_secrets import UserSecretsClient

WANDB_ENTITY = '23f1003053-dl-genai-project'
WANDB_PROJECT = '23f1003053-t32026'
RUN_NAME = 'unet_resnet34_pretrained'
WANDB_MODE = 'online'

try:
    wandb.login(key=UserSecretsClient().get_secret('WANDB_API_KEY'))
except Exception as e:
    WANDB_MODE = 'disabled'
    print('W&B login failed, logging disabled:', repr(e)[:150])

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f1003053 (23f1003053-indian-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 3. Setup, data and split

In [5]:
import os, time, random
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split

DATA_ROOT = '/kaggle/input/competitions/industrial-surface-defect-detection/public'
H, W = 128, 800
N = H * W
CLASSES = ['defect_a', 'defect_b', 'defect_c', 'defect_d']
SEED = 42
QUICK = False
EPOCHS = 20
BATCH = 16

device = 'cuda' if torch.cuda.is_available() else 'cpu'
use_amp = device == 'cuda'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

df = pd.read_csv(f'{DATA_ROOT}/train.csv')
df['mask_rle'] = df['mask_rle'].fillna('').astype(str)
df['has'] = df['mask_rle'].str.strip() != ''
pos = df[df['has']]

img_paths = {}
for root, _, files in os.walk('/kaggle/input'):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
            img_paths[os.path.splitext(f)[0]] = os.path.join(root, f)

labels = pos.groupby('image_id')['defect_type'].apply(lambda s: '+'.join(sorted(s)))
combo = labels.reindex(df['image_id'].drop_duplicates()).fillna('clean')
rle_map = {(r.image_id, r.defect_type): r.mask_rle for r in df.itertuples()}
train_ids, val_ids = train_test_split(combo.index.to_numpy(), test_size=0.2, stratify=combo.values, random_state=SEED)

if QUICK:
    train_ids, val_ids, EPOCHS = train_ids[:320], val_ids[:96], 1

print('device:', device, '| images found:', len(img_paths))
print('train:', len(train_ids), '| val:', len(val_ids), '| epochs:', EPOCHS)

device: cuda | images found: 14393
train: 9114 | val: 2279 | epochs: 20


## 4. Dataset, model, loss and metric

In [6]:
def rle_decode(rle):
    mask = np.zeros(N, dtype=np.uint8)
    if rle.strip():
        t = list(map(int, rle.split()))
        for start, length in zip(t[0::2], t[1::2]):
            a = start - 1
            mask[a:min(a + length, N)] = 1
    return mask.reshape((H, W), order='F')


class SteelDataset(Dataset):
    def __init__(self, ids, train=False):
        self.ids, self.train = list(ids), train

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        iid = self.ids[i]
        img = cv2.imread(img_paths[iid], cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
        img = (img - 0.5) / 0.5
        masks = np.stack([rle_decode(rle_map[(iid, c)]) for c in CLASSES]).astype(np.float32)
        if self.train:
            if random.random() < 0.5:
                img, masks = img[:, ::-1], masks[:, :, ::-1]
            if random.random() < 0.5:
                img, masks = img[::-1], masks[:, ::-1]
        return torch.from_numpy(np.ascontiguousarray(img))[None], torch.from_numpy(np.ascontiguousarray(masks))


train_loader = DataLoader(SteelDataset(train_ids, True), batch_size=BATCH, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
val_loader = DataLoader(SteelDataset(val_ids), batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)


def build_model():
    try:
        m = smp.Unet('resnet34', encoder_weights='imagenet', in_channels=1, classes=4)
        print('pretrained ImageNet encoder loaded')
    except Exception as e:
        print('pretrained weights failed, using random init:', repr(e)[:150])
        m = smp.Unet('resnet34', encoder_weights=None, in_channels=1, classes=4)
    return m.to(device)


bce = nn.BCEWithLogitsLoss()

def dice_loss(logits, target, eps=1e-6):
    p = torch.sigmoid(logits)
    inter = (p * target).sum((2, 3))
    den = p.sum((2, 3)) + target.sum((2, 3))
    return 1 - ((2 * inter + eps) / (den + eps)).mean()


def competition_score(dice_pos, fp_frac_empty, empty_correct):
    fg = float(np.mean(dice_pos)) if len(dice_pos) else 0.0
    fps = 1 - float(np.mean(fp_frac_empty)) if len(fp_frac_empty) else 1.0
    ea = float(np.mean(empty_correct)) if len(empty_correct) else 1.0
    return fg, fps, ea, 0.7 * fg + 0.2 * fps + 0.1 * ea


@torch.no_grad()
def evaluate(model, loader, thresholds):
    model.eval()
    rec = {t: ([], [], []) for t in thresholds}
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.amp.autocast(device_type=device, enabled=use_amp):
            prob = torch.sigmoid(model(x).float())
        ga = y.sum((2, 3))
        for t in thresholds:
            pred = (prob > t).float()
            pa = pred.sum((2, 3))
            den = pa + ga
            d = torch.where(den > 0, 2 * (pred * y).sum((2, 3)) / den.clamp(min=1), torch.ones_like(den))
            rec[t][0].append(d.cpu().numpy()); rec[t][1].append(ga.cpu().numpy()); rec[t][2].append(pa.cpu().numpy())
    return {t: tuple(np.concatenate(v) for v in rec[t]) for t in thresholds}


def summarize(d, ga, pa):
    is_pos = ga > 0
    is_empty = ~is_pos
    fg, fps, ea, score = competition_score(d[is_pos], pa[is_empty] / N, pa[is_empty] == 0)
    return {'fp_rate': float((pa[is_empty] > 0).mean()), 'foreground_dice': fg,
            'naive_mean_dice': float(d.mean()), 'official_score': score}

## 5. Training

In [7]:
model = build_model()
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
best = -1.0

wandb_config = dict(
    architecture='Unet', encoder='resnet34', pretrained=True, in_channels=1,
    epochs=EPOCHS, batch_size=BATCH, lr=3e-4, weight_decay=1e-4,
    scheduler='cosine', loss='BCE+Dice', augmentation='hflip+vflip',
    seed=SEED, n_train=len(train_ids), n_val=len(val_ids),
    split='stratified 80/20 on label combination',
)
try:
    run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name=f'{RUN_NAME}_e{EPOCHS}',
                     group='surfaceguard-segmentation', tags=['unet', 'resnet34', 'pretrained'],
                     config=wandb_config, mode=WANDB_MODE)
except Exception as e:
    print('wandb.init failed, logging disabled:', repr(e)[:200])
    run = wandb.init(mode='disabled')
wandb.define_metric('epoch')
wandb.define_metric('train/*', step_metric='epoch')
wandb.define_metric('val/*', step_metric='epoch')

for epoch in range(EPOCHS):
    t0 = time.time()
    cur_lr = opt.param_groups[0]['lr']
    model.train()
    run_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device, enabled=use_amp):
            logits = model(x)
        loss = bce(logits.float(), y) + dice_loss(logits.float(), y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        run_loss += loss.item() * x.size(0)
    sched.step()

    s = summarize(*evaluate(model, val_loader, [0.5])[0.5])
    train_loss = run_loss / (len(train_loader) * BATCH)
    if s['official_score'] > best:
        best = s['official_score']
        torch.save(model.state_dict(), '/kaggle/working/unet_r34_best.pt')

    wandb.log({
        'epoch': epoch + 1,
        'train/loss': train_loss,
        'val/foreground_dice': s['foreground_dice'],
        'val/fp_rate': s['fp_rate'],
        'val/naive_mean_dice': s['naive_mean_dice'],
        'val/official_score': s['official_score'],
        'lr': cur_lr,
        'time/epoch_s': time.time() - t0,
    })
    print(f"epoch {epoch + 1}/{EPOCHS} | loss {train_loss:.4f} | "
          f"fg_dice {s['foreground_dice']:.3f} | fp_rate {s['fp_rate']:.3f} | "
          f"official {s['official_score']:.4f} | {time.time() - t0:.0f}s")

wandb.summary['best_val_official_score'] = best
try:
    art = wandb.Artifact('unet_r34_best', type='model')
    art.add_file('/kaggle/working/unet_r34_best.pt')
    run.log_artifact(art)
except Exception as e:
    print('artifact upload skipped:', repr(e)[:150])

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

pretrained ImageNet encoder loaded
wandb.init failed, logging disabled: CommError('entity 23f1003053-dl-genai-project not found during upsertBucket')
epoch 1/20 | loss 1.1018 | fg_dice 0.413 | fp_rate 0.215 | official 0.5673 | 154s
epoch 2/20 | loss 0.9721 | fg_dice 0.370 | fp_rate 0.138 | official 0.5447 | 115s
epoch 3/20 | loss 0.9590 | fg_dice 0.502 | fp_rate 0.311 | official 0.6193 | 115s
epoch 4/20 | loss 0.9510 | fg_dice 0.518 | fp_rate 0.353 | official 0.6264 | 115s
epoch 5/20 | loss 0.9449 | fg_dice 0.507 | fp_rate 0.378 | official 0.6158 | 115s
epoch 6/20 | loss 0.9401 | fg_dice 0.562 | fp_rate 0.409 | official 0.6519 | 115s
epoch 7/20 | loss 0.9352 | fg_dice 0.568 | fp_rate 0.365 | official 0.6604 | 115s
epoch 8/20 | loss 0.9311 | fg_dice 0.585 | fp_rate 0.316 | official 0.6771 | 115s
epoch 9/20 | loss 0.9285 | fg_dice 0.601 | fp_rate 0.352 | official 0.6854 | 115s
epoch 10/20 | loss 0.9245 | fg_dice 0.606 | fp_rate 0.341 | official 0.6894 | 115s
epoch 11/20 | loss 0.9226 | f

## 6. Threshold sweep (Q12 table)

In [8]:
model.load_state_dict(torch.load('/kaggle/working/unet_r34_best.pt', map_location=device))
THRESHOLDS = [0.3, 0.5, 0.7, 0.9]
ev = evaluate(model, val_loader, THRESHOLDS)

table = pd.DataFrame({t: summarize(*ev[t]) for t in THRESHOLDS}).T
table.index.name = 'threshold'
print(table.round(4))
print('holdout empty-pair fraction:', round(float((ev[0.5][1] == 0).mean()), 4))
print('peak tau (naive mean Dice):', table['naive_mean_dice'].idxmax())
print('peak tau (official score):', table['official_score'].idxmax())

d, ga, pa = ev[0.5]
for j, c in enumerate(CLASSES):
    p = ga[:, j] > 0
    print(c, '| positives:', int(p.sum()), '| fg dice:', round(float(d[:, j][p].mean()), 3) if p.any() else 'n/a')

if wandb.run is not None:
    wandb.log({'threshold_sweep': wandb.Table(dataframe=table.reset_index())})
    for j, c in enumerate(CLASSES):
        p = ga[:, j] > 0
        wandb.summary[f'val/fg_dice_{c}'] = float(d[:, j][p].mean()) if p.any() else 0.0

           fp_rate  foreground_dice  naive_mean_dice  official_score
threshold                                                           
0.3         0.2625           0.6357           0.7214          0.7184
0.5         0.2283           0.6328           0.7498          0.7199
0.7         0.1917           0.6244           0.7793          0.7177
0.9         0.1393           0.5941           0.8186          0.7018
holdout empty-pair fraction: 0.8419
peak tau (naive mean Dice): 0.9
peak tau (official score): 0.5
defect_a | positives: 216 | fg dice: 0.437
defect_b | positives: 63 | fg dice: 0.455
defect_c | positives: 820 | fg dice: 0.654
defect_d | positives: 342 | fg dice: 0.738


## 7. Prediction helpers (RLE encoder and 4-flip TTA)

In [9]:
def rle_encode(mask):
    pixels = np.concatenate([[0], mask.flatten(order='F'), [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(map(str, runs))


def predict_probs(model, x):
    with torch.amp.autocast(device_type=device, enabled=use_amp):
        p = torch.sigmoid(model(x).float())
        p = p + torch.sigmoid(model(x.flip(3)).float()).flip(3)
        p = p + torch.sigmoid(model(x.flip(2)).float()).flip(2)
        p = p + torch.sigmoid(model(x.flip((2, 3))).float()).flip((2, 3))
    return p / 4

## 8. RLE round-trip check on validation

In [10]:
model.eval()
ds = SteelDataset(val_ids[:300])
dices = []
with torch.no_grad():
    for i in range(len(ds)):
        x, y = ds[i]
        p = predict_probs(model, x[None].to(device))[0].cpu().numpy()
        for k in range(4):
            gt = y[k].numpy().astype(np.uint8)
            if gt.sum() == 0:
                continue
            m = (p[k] > 0.5).astype(np.uint8)
            back = rle_decode(rle_encode(m) if m.any() else ' ')
            dices.append(2 * (back & gt).sum() / (back.sum() + gt.sum()))
print('round-trip foreground dice:', round(float(np.mean(dices)), 4), '| positive pairs:', len(dices))

round-trip foreground dice: 0.6399 | positive pairs: 197


## 9. Per-class threshold and minimum-area tuning

In [11]:
import json, gc

@torch.no_grad()
def collect_val_probs(model, loader):
    model.eval()
    P, Y = [], []
    for x, y in loader:
        P.append(predict_probs(model, x.to(device)).half().cpu().numpy())
        Y.append(y.numpy().astype(np.uint8))
    return np.concatenate(P), np.concatenate(Y)


P, Y = collect_val_probs(model, val_loader)
TAUS = np.round(np.arange(0.30, 0.91, 0.05), 2)
MINAREAS = [0, 50, 100, 200, 400, 800, 1500]

gt_any = Y.sum((2, 3)) > 0
n_pos_total = int(gt_any.sum())
n_empty_total = int(gt_any.size - n_pos_total)

best_params, total, base_total = {}, 0.2, 0.2
for k, c in enumerate(CLASSES):
    gt = Y[:, k].astype(bool)
    gt_area = gt.sum((1, 2))
    scores = {}
    for t in TAUS:
        acc = {ma: [0.0, 0.0, 0.0] for ma in MINAREAS}
        for i in range(len(P)):
            m = (P[i, k] > t).astype(np.uint8)
            if not m.any():
                if gt_area[i] == 0:
                    for ma in MINAREAS:
                        acc[ma][2] += 1
                continue
            n, lab, st, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
            areas = st[:, cv2.CC_STAT_AREA]
            for ma in MINAREAS:
                if ma > 0:
                    keep = areas >= ma
                    keep[0] = False
                    mm = keep[lab]
                else:
                    mm = m.astype(bool)
                pa = int(mm.sum())
                if gt_area[i] > 0:
                    acc[ma][0] += 2 * np.logical_and(mm, gt[i]).sum() / (pa + gt_area[i])
                else:
                    acc[ma][1] += pa / N
                    acc[ma][2] += pa == 0
        for ma in MINAREAS:
            d_sum, fp_sum, ok_sum = acc[ma]
            scores[(t, ma)] = 0.7 * d_sum / n_pos_total - 0.2 * fp_sum / n_empty_total + 0.1 * ok_sum / n_empty_total
    (bt, bma), bs = max(scores.items(), key=lambda kv: kv[1])
    best_params[c] = (float(bt), int(bma))
    total += bs
    base_total += scores[(0.5, 0)]
    rate = float((P[:, k] > 0.5).reshape(len(P), -1).any(1).mean())
    print(c, '| best tau:', bt, '| min area:', bma, '| val pred-positive rate @0.5:', round(rate, 3),
          '| true rate:', round(float(gt.reshape(len(P), -1).any(1).mean()), 3))

print('validation score, tau=0.5 and no filter:', round(base_total, 4))
print('validation score, tuned per class:', round(total, 4))
json.dump(best_params, open('/kaggle/working/best_params.json', 'w'))
print(best_params)

if wandb.run is not None:
    wandb.summary['val/score_untuned'] = base_total
    wandb.summary['val/score_tuned'] = total
    wandb.config.update({'tuned_params': best_params}, allow_val_change=True)
    wandb.finish()

del P, Y
gc.collect()

defect_a | best tau: 0.3 | min area: 100 | val pred-positive rate @0.5: 0.295 | true rate: 0.095
defect_b | best tau: 0.35 | min area: 200 | val pred-positive rate @0.5: 0.149 | true rate: 0.028
defect_c | best tau: 0.35 | min area: 100 | val pred-positive rate @0.5: 0.542 | true rate: 0.36
defect_d | best tau: 0.3 | min area: 200 | val pred-positive rate @0.5: 0.199 | true rate: 0.15
validation score, tau=0.5 and no filter: 0.7322
validation score, tuned per class: 0.7414
{'defect_a': (0.3, 100), 'defect_b': (0.35, 200), 'defect_c': (0.35, 100), 'defect_d': (0.3, 200)}


168

## 10. Inference and submission

In [12]:
DATA_ROOT = '/kaggle/input/competitions/industrial-surface-defect-detection/public'
WEIGHTS = '/kaggle/working/unet_r34_best.pt'
IMG_EXT = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')

params = json.load(open('/kaggle/working/best_params.json'))
THR = {c: params[c][0] for c in CLASSES}
MIN_AREA = {c: params[c][1] for c in CLASSES}

sub = pd.read_csv(f'{DATA_ROOT}/sample_submission.csv')
test_ids = sub['image_id'].drop_duplicates().tolist()
test_set = set(test_ids)

test_paths = {}
for root, _, files in os.walk('/kaggle/input'):
    for f in files:
        stem, ext = os.path.splitext(f)
        if ext.lower() in IMG_EXT and stem in test_set:
            test_paths[stem] = os.path.join(root, f)
print('test images:', len(test_ids), '| found:', len(test_paths))
assert len(test_paths) == len(test_ids)

model = smp.Unet('resnet34', encoder_weights=None, in_channels=1, classes=4).to(device)
model.load_state_dict(torch.load(WEIGHTS, map_location=device))
model.eval()


def clean_mask(m, min_area):
    if min_area <= 0 or not m.any():
        return m
    n, lab, st, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    keep = st[:, cv2.CC_STAT_AREA] >= min_area
    keep[0] = False
    return keep[lab].astype(np.uint8)


rle_out = {}
with torch.no_grad():
    for i in range(0, len(test_ids), 32):
        ids = test_ids[i:i + 32]
        batch = np.stack([(cv2.imread(test_paths[j], cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0 - 0.5) / 0.5 for j in ids])
        probs = predict_probs(model, torch.from_numpy(batch)[:, None].to(device)).cpu().numpy()
        for b, iid in enumerate(ids):
            for k, c in enumerate(CLASSES):
                m = clean_mask((probs[b, k] > THR[c]).astype(np.uint8), MIN_AREA[c])
                rle_out[(iid, c)] = rle_encode(m) if m.any() else ' '

sub['mask_rle'] = [rle_out[(i, d)] for i, d in zip(sub['image_id'], sub['defect_type'])]
sub.to_csv('submission.csv', index=False)

print(sub.shape)
print((sub['mask_rle'].str.strip() != '').groupby(sub['defect_type']).mean().round(3))

test images: 3000 | found: 3000
(12000, 4)
defect_type
defect_a    0.328
defect_b    0.130
defect_c    0.563
defect_d    0.385
Name: mask_rle, dtype: float64
